# 03 — Label Exercise Phases

Auto-detect movement cycles and label each frame with a phase value (0.0 - 1.0).

## Phase meaning
- **Cyclic exercises** (push-ups, lunges): `0.0` = start/extended, `0.5` = deepest point, `1.0` = back to start
- **Static exercises** (plank): ~`0.5` constant with minor breathing variation

## How it works
1. Computes the primary joint angle per frame (e.g., elbow angle for push-ups)
2. Smooths the signal with Savitzky-Golay filter
3. Finds peaks (extended) and valleys (deepest) using `scipy.signal.find_peaks`
4. Interpolates phase between detected keypoints
5. Assigns rep numbers based on phase wrapping

## Inputs
```
data/extracted/*.npz   — landmarks (N, 33, 3)
data/extracted/*.json  — metadata (exercise type)
```

## Outputs
```
data/labeled/*_labeled.npz   — landmarks + phases + rep_numbers + primary_angles
data/labeled/*_labeled.json  — labeling metadata
```

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

from src.phase_labeler import (
    label_video, save_labels,
    compute_angle_series, smooth_signal,
    EXERCISE_ANGLES,
)

%matplotlib inline
plt.rcParams['figure.figsize'] = (16, 6)

## Label all extracted videos

In [ ]:
extracted_dir = Path('../data/extracted')
npz_files = sorted(extracted_dir.glob('*.npz'))

all_results = []

for npz_file in npz_files:
    data = np.load(npz_file)
    landmarks = data['landmarks']  # (N, 33, 3)
    
    json_file = npz_file.with_suffix('.json')
    with open(json_file) as f:
        meta = json.load(f)
    exercise = meta['exercise']
    video_name = npz_file.stem
    
    print(f'\nLabeling: {video_name} ({exercise})')
    
    labels = label_video(landmarks, exercise)
    save_labels(labels, landmarks, exercise, video_name, '../data/labeled')
    
    phases = np.array([l.phase for l in labels])
    reps = max(l.rep_number for l in labels) + 1 if exercise != 'plank' else 0
    print(f'  Frames: {len(labels)}, Reps detected: {reps}')
    print(f'  Phase range: [{phases.min():.2f}, {phases.max():.2f}]')
    
    all_results.append({
        'video_name': video_name,
        'exercise': exercise,
        'labels': labels,
        'landmarks': landmarks,
    })

print(f'\nLabeled {len(all_results)} video(s).')

## Visualize: Angle curves with phase overlay

For each video:
- **Top plot:** Raw and smoothed angle signal. Peaks = extended position, valleys = deepest.
- **Bottom plot:** Resulting phase labels (0.0 -> 0.5 -> 1.0 per rep).
- **Red dashed lines:** Rep boundaries.

Verify that peaks/valleys align correctly with the exercise motion.

In [ ]:
for result in all_results:
    labels = result['labels']
    landmarks = result['landmarks']
    exercise = result['exercise']
    video_name = result['video_name']
    
    angles = compute_angle_series(landmarks, exercise)
    smoothed = smooth_signal(angles, window=7)
    phases = np.array([l.phase for l in labels])
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
    
    ax1.plot(angles, alpha=0.4, label='Raw angle', color='gray')
    ax1.plot(smoothed, label='Smoothed', color='blue', linewidth=2)
    ax1.set_ylabel('Angle (degrees)')
    ax1.set_title(f'{video_name} — {exercise} — Primary Angle')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(phases, color='green', linewidth=2)
    ax2.fill_between(range(len(phases)), phases, alpha=0.2, color='green')
    ax2.set_ylabel('Phase (0-1)')
    ax2.set_xlabel('Frame')
    ax2.set_title(f'{video_name} — Exercise Phase')
    ax2.set_ylim(-0.05, 1.05)
    ax2.grid(True, alpha=0.3)
    
    if exercise != 'plank':
        rep_numbers = np.array([l.rep_number for l in labels])
        for i in range(1, len(rep_numbers)):
            if rep_numbers[i] != rep_numbers[i-1]:
                ax1.axvline(i, color='red', alpha=0.5, linestyle='--')
                ax2.axvline(i, color='red', alpha=0.5, linestyle='--')
    
    plt.tight_layout()
    plt.show()

## Manual review

If phase labels look wrong, you can manually adjust here.

Check that:
1. Phase 0.0 aligns with start/extended position
2. Phase 0.5 aligns with deepest/contracted position
3. Rep boundaries are clean
4. Smooth transitions between phases

In [ ]:
# Manual correction example (uncomment and modify as needed):
#
# result = all_results[0]
# labels = result['labels']
#
# for i in range(100, 150):
#     labels[i].phase = 0.5
#
# save_labels(
#     labels, result['landmarks'], result['exercise'],
#     result['video_name'], '../data/labeled'
# )

print('Phase labeling complete. Review plots above for quality.')

---
**Next:** Run `04_train_model.ipynb` to train the pose correction model.